# fio DFS controlled sweep — tail latency (Aurora, 1 file per job)

`rand_write`, 1 MiB blocks, single Aurora node, DAOS `dfs` ioengine.
The sweep holds **aggregate queue depth fixed at 256** (`numjobs x iodepth = 256`)
and a fixed **8 GiB** working set, while varying how that concurrency is split
across jobs. Each job owns **exactly one DFS object** (`nrfiles=1`), so `numjobs`
is also the number of distinct objects the load is spread over — that is the
variable this campaign is designed to isolate.

Each run: 60 s measured, 10 s ramp, `direct=1`, `group_reporting=1`.

The loader below reads the most recent campaign directory under this folder and
aggregates by `(numjobs, iodepth)`, so re-running it after a new campaign lands
refreshes every figure without edits. Set `CAMPAIGNS = None` to pool all campaigns
instead — off by default so a figure never mixes runs from different jobs and
different compute nodes.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FuncFormatter

RESULTS_DIR = Path(".")

PCT_KEYS   = ["50.000000", "90.000000", "95.000000", "99.000000", "99.900000", "99.990000"]
PCT_LABELS = ["p50", "p90", "p95", "p99", "p99.9", "p99.99"]
PCT_VALUES = [50.0, 90.0, 95.0, 99.0, 99.9, 99.99]

FNAME_RE = re.compile(r"fio_r(\d+)_p(\d+)_bs(\S+?)_nj(\d+)_iod(\d+)_(\d+)\.json$")

# Which campaign directories feed the figures.
#   "latest" - the most recent campaign only (default): a figure is never a mix of
#              runs from different jobs and different compute nodes.
#   None     - pool every campaign found under RESULTS_DIR.
#   [...]    - an explicit list of campaign directory names.
CAMPAIGNS = "latest"

## Load results

fio writes its JSON only when a run completes, so a campaign killed mid-flight
leaves zero-byte or truncated files behind. Those are skipped and reported rather
than silently dropped.

In [ ]:
def select_campaigns(root, which):
    found = sorted(p.name for p in Path(root).iterdir()
                   if p.is_dir() and any(p.glob("fio_r*.json")))
    if which is None:
        return found
    if which == "latest":
        return found[-1:]
    return [c for c in found if c in which]


def load_runs(root, campaigns):
    rows, skipped = [], []
    for f in sorted(Path(root).glob("*/fio_r*.json")):
        if f.parent.name not in campaigns:
            continue
        m = FNAME_RE.search(f.name)
        if m is None:
            continue
        if f.stat().st_size == 0:
            skipped.append((f, "empty"))
            continue
        try:
            d = json.loads(f.read_text())
        except json.JSONDecodeError:
            skipped.append((f, "truncated"))
            continue

        job = d["jobs"][0]
        w = job["write"]
        clat, lat, slat = w["clat_ns"], w["lat_ns"], w["slat_ns"]
        ms = lambda v: v / 1e6

        row = dict(
            campaign      = f.parent.name,
            repeat        = int(m.group(1)),
            block_size    = m.group(3),
            numjobs       = int(m.group(4)),
            iodepth       = int(m.group(5)),
            bw_GiBs       = w["bw"] / 1024 / 1024,
            iops          = w["iops"],
            slat_mean_ms  = ms(slat["mean"]),
            clat_min_ms   = ms(clat["min"]),
            clat_mean_ms  = ms(clat["mean"]),
            clat_stdev_ms = ms(clat["stddev"]),
            clat_max_ms   = ms(clat["max"]),
            lat_min_ms    = ms(lat["min"]),
            lat_mean_ms   = ms(lat["mean"]),
            lat_max_ms    = ms(lat["max"]),
            usr_cpu       = job["usr_cpu"],
            sys_cpu       = job["sys_cpu"],
        )
        for name, blk in (("clat", clat), ("slat", slat), ("lat", lat)):
            pct = blk.get("percentile")
            for key, lab in zip(PCT_KEYS, PCT_LABELS):
                row[f"{name}_{lab}_ms"] = ms(pct[key]) if pct else float("nan")
        rows.append(row)

    if not rows:
        raise SystemExit(
            f"No completed fio JSON found under {Path(root).resolve()}.\n"
            "Run this notebook from the campaign root (the directory holding the "
            "<timestamp>/ subdirectories)."
        )
    df = pd.DataFrame(rows).sort_values(["numjobs", "campaign", "repeat"]).reset_index(drop=True)
    return df, skipped


selected = select_campaigns(RESULTS_DIR, CAMPAIGNS)
available = select_campaigns(RESULTS_DIR, None)

df_runs, skipped = load_runs(RESULTS_DIR, selected)

print(f"Campaigns used    : {', '.join(selected)}")
if set(available) - set(selected):
    print(f"Campaigns ignored : {', '.join(sorted(set(available) - set(selected)))} "
          f"(set CAMPAIGNS = None to pool them)")
print(f"Loaded {len(df_runs)} completed runs, "
      f"{df_runs.groupby(['numjobs','iodepth']).ngroups} configs, "
      f"{df_runs.groupby(['numjobs','iodepth']).size().min()}-"
      f"{df_runs.groupby(['numjobs','iodepth']).size().max()} repeats each")
for f, why in skipped:
    print(f"  skipped ({why}): {f.parent.name}/{f.name}")

df_runs[["campaign", "repeat", "numjobs", "iodepth", "bw_GiBs", "iops",
         "clat_p50_ms", "clat_p99_ms", "clat_p99.99_ms", "clat_max_ms"]]

## Aggregate by configuration

Median across repeats; the spread column reports the full range when more than one
repeat exists for a config.

In [ ]:
METRICS = [c for c in df_runs.columns
           if c not in ("campaign", "repeat", "block_size", "numjobs", "iodepth")]

grp    = df_runs.groupby(["numjobs", "iodepth"])
df     = grp[METRICS].median()
df["n_runs"] = grp.size()

order  = lambda t: t.reset_index().sort_values("numjobs").reset_index(drop=True)
df     = order(df)
df_std = order(grp[METRICS].std())
df_lo  = order(grp[METRICS].min())
df_hi  = order(grp[METRICS].max())

df["label"]  = df.apply(lambda r: f"nj={int(r.numjobs)}\niod={int(r.iodepth)}", axis=1)
df["series"] = df.apply(lambda r: f"nj={int(r.numjobs)}, iod={int(r.iodepth)}", axis=1)

summary = df[["series", "n_runs", "bw_GiBs", "iops",
              "lat_mean_ms", "lat_max_ms", "slat_mean_ms"]
             + [f"clat_{l}_ms" for l in PCT_LABELS] + ["clat_max_ms"]].copy()
summary.columns = ["config", "runs", "BW (GiB/s)", "IOPS",
                   "lat mean (ms)", "lat max (ms)", "slat mean (ms)"] \
                  + [f"clat {l} (ms)" for l in PCT_LABELS] + ["clat max (ms)"]
summary.style.format({c: "{:,.2f}" for c in summary.columns if c not in ("config", "runs")}) \
       .format({"IOPS": "{:,.0f}"}) \
       .hide(axis="index")

## Plot style

Categorical hues are assigned in fixed slot order (one per configuration, never
recycled); the percentile ramp is a single blue hue, light to dark.

In [ ]:
SERIES  = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]   # categorical slots 1-5
BLUES   = ["#86b6ef", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#104281"]  # ordinal ramp
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 200, "font.size": 9,
    "axes.edgecolor": GRID, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "savefig.bbox": "tight",
})

# Tail axis: percentile p plotted at 1/(1-p), so each decade of tail gets equal width.
def tail_x(p):
    return 1.0 / (1.0 - p / 100.0)

XS      = [tail_x(p) for p in PCT_VALUES]
X_MAX   = XS[-1] * 4          # slot for the observed maximum, past p99.99
COLOR_OF = {r.series: SERIES[i % len(SERIES)] for i, r in df.iterrows()}


def tail_axis(ax, with_max=True):
    ticks  = XS + ([X_MAX] if with_max else [])
    labels = PCT_LABELS + (["max"] if with_max else [])
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_minor_locator(FixedLocator([]))
    ax.set_xticklabels(labels)
    ax.set_xlim(XS[0] / 1.5, ticks[-1] * 1.5)
    ax.set_xlabel("Completion-latency percentile")
    ax.grid(axis="y", color=GRID, lw=0.7)
    ax.set_axisbelow(True)

## Tail latency curve

The main view: completion latency against percentile, with the x-axis stretched
so each decade of the tail (p90 → p99 → p99.9 → p99.99) occupies equal width. A
flat line means the distribution is tight; a line that turns upward on the right
is where the tail lives. The rightmost point is the single worst I/O observed in
the run, not a percentile — it is plotted hollow and separated by a dashed segment.

Where a configuration has more than one repeat, the shaded band spans the full
min-to-max range across those repeats and the line is their median. Configurations
with a single run are drawn as a bare line, and the legend gives each one's `n`.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))

for i, r in df.iterrows():
    ys = [r[f"clat_{lab}_ms"] for lab in PCT_LABELS]
    c  = COLOR_OF[r.series]
    if r.n_runs > 1:
        lo = [df_lo.loc[i, f"clat_{lab}_ms"] for lab in PCT_LABELS]
        hi = [df_hi.loc[i, f"clat_{lab}_ms"] for lab in PCT_LABELS]
        ax.fill_between(XS, lo, hi, color=c, alpha=0.15, lw=0)
    ax.plot(XS, ys, "-o", color=c, lw=2, ms=6, mec="white", mew=1.2,
            label=f"{r.series}  (n={int(r.n_runs)})")
    ax.plot([XS[-1], X_MAX], [ys[-1], r.clat_max_ms], "--", color=c, lw=1.2, alpha=0.7)
    ax.plot([X_MAX], [r.clat_max_ms], "o", color="white", mec=c, mew=2, ms=7)

tail_axis(ax)
ax.set_yscale("log")
ax.yaxis.set_major_locator(FixedLocator(
    [0.2, 0.5, 1, 2, 3, 5, 10, 20, 30, 50, 100, 200, 500, 1000, 2000, 5000]))
ax.yaxis.set_minor_locator(FixedLocator([]))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
band_top = max(df_hi[f"clat_{lab}_ms"].max() for lab in PCT_LABELS)
ax.set_ylim(df_lo["clat_p50_ms"].min() / 1.5,
            max(band_top, df["clat_max_ms"].max()) * 1.35)
ax.set_ylabel("Completion latency (ms)")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), ncols=3, fontsize=8,
          columnspacing=1.4, handlelength=1.6)

fig.savefig(RESULTS_DIR / "tail_latency_curve.pdf")
plt.show()

## Percentiles side by side

The same numbers as bars, which makes the absolute gap between configurations
easier to read off than the log axis above.

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 3.6))

n = len(PCT_LABELS)
w = 0.86 / n
x = np.arange(len(df))

multi = (df["n_runs"] > 1).values

for k, lab in enumerate(PCT_LABELS):
    off = (k - (n - 1) / 2) * w
    col = f"clat_{lab}_ms"
    med = df[col].values
    # Whiskers span min-to-max across repeats; zero-length for single-run configs.
    err = np.vstack([np.where(multi, med - df_lo[col].values, 0),
                     np.where(multi, df_hi[col].values - med, 0)])
    ax.bar(x + off, med, w * 0.88, label=lab,
           color=BLUES[k], edgecolor="white", lw=0.6,
           yerr=err, capsize=2,
           error_kw=dict(elinewidth=0.8, ecolor=INK2, capthick=0.8))

ax.set_xticks(x)
ax.set_xticklabels(df["label"])
ax.set_ylabel("Completion latency (ms)")
ax.grid(axis="y", color=GRID, lw=0.7)
ax.set_axisbelow(True)
ax.margins(y=0.12)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), ncols=6, fontsize=8,
          columnspacing=1.2, handlelength=1.2)

fig.savefig(RESULTS_DIR / "tail_latency_bars.pdf")
plt.show()

## Submit vs complete latency, median vs tail

`slat` is the time to submit an I/O to the DFS engine; `clat` is the time from
submission to completion. Both are shown at p50 and p99.99, grouped by
configuration. Hue separates the two components, hatching separates median from
tail.

The y-axis is linear. Because slat runs in tens of microseconds while the clat tail
runs in tens to hundreds of milliseconds, the slat bars are flat at this scale —
every bar carries its value as a label, which is the only way to read them here.

Bars are medians across repeats with no whiskers: a single `nj=16` run reached a
1.4 s clat p99.99, and drawing that range would compress every other bar to
invisibility. The run-to-run spread is in the variability table and in the bands on
the tail-latency curve instead.

In [ ]:
BAR_SPECS = [
    ("slat_p50_ms",    "slat p50",    SERIES[0], ""),
    ("slat_p99.99_ms", "slat p99.99", SERIES[0], "///"),
    ("clat_p50_ms",    "clat p50",    SERIES[1], ""),
    ("clat_p99.99_ms", "clat p99.99", SERIES[1], "///"),
]

def fmt_val(v):
    if v >= 100:  return f"{v:,.0f}"
    if v >= 1:    return f"{v:.1f}"
    return f"{v:.3f}"

fig, ax = plt.subplots(figsize=(7.4, 4.0))

n = len(BAR_SPECS)
w = 0.84 / n
x = np.arange(len(df))
multi = (df["n_runs"] > 1).values

# Medians only: a single run's 1.4 s clat tail would otherwise set the scale and
# flatten every other bar. Run-to-run spread lives in the CV table and the bands
# on the tail-latency curve.
for k, (col, lab, color, hatch) in enumerate(BAR_SPECS):
    off = (k - (n - 1) / 2) * w
    med = df[col].values
    ax.bar(x + off, med, w * 0.9, label=lab, color=color, hatch=hatch,
           edgecolor="white", lw=0.7)
    for xi, v in zip(x + off, med):
        ax.text(xi, v, "  " + fmt_val(v), ha="center", va="bottom",
                fontsize=7, color=INK2, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(df["label"])
ax.set_ylabel("Latency (ms)")
ax.grid(axis="y", color=GRID, lw=0.7)
ax.set_axisbelow(True)
ax.margins(y=0.30)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), ncols=4, fontsize=8,
          columnspacing=1.4, handlelength=1.6)

fig.savefig(RESULTS_DIR / "slat_clat_p50_p9999.pdf")
plt.show()

## Tail amplification

Each percentile divided by that configuration's own p50. This separates *how heavy
the tail is* from *how fast the configuration is overall* — a config can be fast at
the median and still have a tail several times worse than its own typical I/O.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.6))

worst = None
for _, r in df.iterrows():
    ys = [r[f"clat_{lab}_ms"] / r["clat_p50_ms"] for lab in PCT_LABELS]
    c  = COLOR_OF[r.series]
    ax.plot(XS, ys, "-o", color=c, lw=2, ms=6, mec="white", mew=1.2, label=r.series)
    if worst is None or ys[-1] > worst[0]:
        worst = (ys[-1], c)

ax.axhline(1.0, color=GRID, lw=1)
ax.annotate(f"{worst[0]:.1f}x", (XS[-1], worst[0]), textcoords="offset points",
            xytext=(8, 0), va="center", fontsize=8, color=INK2)

tail_axis(ax, with_max=False)
ax.set_ylabel("Latency / p50")
ax.margins(y=0.12)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), ncols=3, fontsize=8,
          columnspacing=1.4, handlelength=1.6)

fig.savefig(RESULTS_DIR / "tail_amplification.pdf")
plt.show()

## Throughput against latency

Two measures on two separate panels rather than one twin-axis plot, so neither
scale is misread against the other.

The latency panels use **total** latency (`lat_ns`) — submission plus completion,
i.e. what the application actually waits — rather than `clat` or `slat`. Mean is
what a typical I/O costs; max is the single worst I/O in the run. They get separate
panels because max runs 5-70x mean, and sharing one linear axis would flatten the
mean bars to nothing.

Bandwidth bars carry min-to-max whiskers across repeats. The latency panels are
medians only: one `nj=16` run reached a 3.8 s max, and drawing that range would
compress every other bar to invisibility. That spread is in the variability table
below.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.2, 3.3))

multi = (df["n_runs"] > 1).values

# Bandwidth keeps its min-to-max whiskers; the latency panels are medians only,
# because one nj=16 run reached a 3.8 s max and its range would flatten the rest.
panels = [
    (axes[0], "bw_GiBs",     "Write bandwidth (GiB/s)", "Throughput",   SERIES[0], "{:,.1f}", True),
    (axes[1], "lat_mean_ms", "Mean total latency (ms)", "Mean latency", SERIES[1], "{:.1f}",  False),
    (axes[2], "lat_max_ms",  "Max total latency (ms)",  "Max latency",  SERIES[2], "{:.1f}",  False),
]

for ax, col, ylab, title, color, fmt, with_err in panels:
    med = df[col].values
    err = None
    if with_err:
        err = np.vstack([np.where(multi, med - df_lo[col].values, 0),
                         np.where(multi, df_hi[col].values - med, 0)])
    ax.bar(df["label"], med, 0.55, color=color, edgecolor="white", lw=0.6,
           yerr=err, capsize=3,
           error_kw=dict(elinewidth=0.9, ecolor=INK2, capthick=0.9))
    tops = med + (err[1] if err is not None else 0)
    for xi, (v, top) in enumerate(zip(med, tops)):
        ax.text(xi, top, fmt.format(v), ha="center", va="bottom", fontsize=8, color=INK2)
    ax.set_ylabel(ylab)
    ax.set_title(title, loc="left", fontsize=10)
    ax.grid(axis="y", color=GRID, lw=0.7)
    ax.set_axisbelow(True)
    ax.margins(y=0.18)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "bandwidth_vs_tail.pdf")
plt.show()

## Run-to-run variability

Only meaningful once a config has more than one repeat; until then this table is
informational and the medians above are single observations.

In [ ]:
CV_METRICS = ["bw_GiBs", "iops", "lat_mean_ms", "lat_max_ms"] \
             + [f"clat_{l}_ms" for l in PCT_LABELS] + ["clat_max_ms"]

if df["n_runs"].max() > 1:
    cv = (df_std[CV_METRICS] / df[CV_METRICS] * 100)
    cv.insert(0, "config", df["series"])
    cv.insert(1, "runs", df["n_runs"].astype(int))

    single = df["n_runs"] <= 1
    if single.any():
        print("No spread available (single run): "
              + "; ".join(df.loc[single, "series"]))

    display(cv.style.hide(axis="index")
              .format({c: "{:.2f}%" for c in CV_METRICS}, na_rep="n/a")
              .map(lambda v: "background-color:#fde2dd"
                             if isinstance(v, float) and v == v and v > 5 else "",
                   subset=CV_METRICS))
else:
    print(f"Single repeat per config ({int(df['n_runs'].max())} run each) - "
          "no variability estimate available yet.")

## What the five runs show

Campaign `20260806_054458`: 3 configurations x 5 repeats, all 15 runs completed.
`nj=32` and `nj=64` are absent because they reproducibly take the compute node down;
that is a separate open issue, not a gap in this campaign.

**Submit latency is negligible, so total latency is completion latency.** `slat` p50
sits at 15-18 microseconds and its p99.99 never exceeds 0.09 ms - under 1% of even
the fastest configuration's median clat. Every latency conclusion below is really a
statement about completion time.

**Everything improves with more jobs except the stability of the tail.** Going from
`nj=4` to `nj=16` raises bandwidth 4x (22.3 -> 89.2 GiB/s) and cuts mean latency 4.2x
(11.06 -> 2.66 ms) and median clat 4.7x (11.86 -> 2.54 ms). On medians alone `nj=16`
wins outright.

**But `nj=16`'s tail is both enormous and unpredictable.** Its median worst-case I/O
is 313 ms, against 13.5 ms for `nj=4`. Worse, it does not repeat: across five
identical runs its clat p99.99 ranged from 8.4 ms to 1367 ms, and its worst I/O from
15.5 ms to 3.81 s. That is a run-to-run CV of 1072% on p99.99 and 518% on max - the
other two configurations sit under 6%. So `nj=16` is not "a configuration with a
heavy tail"; it is a configuration whose tail behaviour is a coin flip.

**`nj=8, iod=32` is the defensible operating point.** It doubles `nj=4`'s bandwidth
(47.6 vs 22.3 GiB/s), halves its mean latency (5.11 vs 11.06 ms), and has the lowest
maximum latency of all three (10.8 ms) - and unlike `nj=16` it is reproducible, with
tail CV under 5%.

**Caveats.** Bandwidth itself is noisy at the two lower job counts (CV 17.5% at
`nj=4`, 15.7% at `nj=8`, against 3.4% at `nj=16`), so the throughput ordering is
solid but the exact values are not. And object count still tracks `numjobs`, so none
of this separates I/O concurrency from object-level parallelism - see the note below.

## Notes on reading these results

`numjobs` is not a pure concurrency knob in this campaign. Because `nrfiles=1`,
raising `numjobs` also raises the number of distinct DFS objects the 8 GiB working
set is striped over, while the aggregate queue depth stays at 256. Throughput and
latency therefore move together with object count, and the tail behaviour should be
attributed to per-object concurrency, not to fio's job count on its own. The
companion `...-controlled-64files` campaign holds the object count fixed instead
and is the comparison that separates the two effects.